# GTA V forecast: exploratory deep-model bake-off (Colab GPU)

**Exploratory, run after the freeze on pre-launch data only. Does not change the pre-registered test or its verdict.**

Runs `exploratory/bakeoff/run_bakeoff.py` (monthly, then daily) on an NVIDIA GPU and builds `REPORT.md`. The plan,
committed before any result, is `exploratory/bakeoff/PLAN.md`.

1. *Runtime > Change runtime type* > any GPU (T4, L4 or A100), then *Connect*. The notebook does not pin a GPU type.
2. Optionally keep `USE_DRIVE = True` below, so finished fits survive a disconnect.
3. *Runtime > Run all.* After a disconnect, just *Run all* again: finished models and cached fits are skipped.

No tokens or secrets are needed: the repository is public and nothing is pushed.

In [ ]:
# 1. GPU check: stop here if no NVIDIA GPU is attached
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else r.stderr or "nvidia-smi not found")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU attached. Use Runtime > Change runtime type > T4 GPU (or better), then Runtime > Run all.")
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
# Settings
USE_DRIVE = True                                     # write results to Google Drive so a disconnect loses nothing
DRIVE_DIR = "/content/drive/MyDrive/gtavi_bakeoff"   # results go to DRIVE_DIR/results, report to DRIVE_DIR/REPORT.md
MODELS = []                                          # [] = all six; or e.g. ["nhits", "tft"]

In [ ]:
# 2. Clone the repository on branch exploratory/bakeoff
import os
REPO = "/content/gtavi_effect"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", "exploratory/bakeoff", "https://github.com/iannh10/gtavi_effect.git", REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "exploratory/bakeoff"], check=True)
subprocess.run(["git", "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Install requirements + neuralforecast, keeping Colab's CUDA build of torch.
#    pandas is pinned to the freeze's version (MANIFEST) so the exact seasonal-naive gate scores the same way.
import sys
with open("/content/keep_torch.txt", "w") as f:
    print(f"torch=={torch.__version__}", file=f)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "neuralforecast==3.3.1", "pandas==3.0.5",
                "-c", "/content/keep_torch.txt"], check=True)
print(subprocess.run([sys.executable, "-c",
    "import sys, torch, neuralforecast, pandas, numpy; print('python', sys.version.split()[0], '| torch', torch.__version__, "
    "'| cuda', torch.cuda.is_available(), '| neuralforecast', neuralforecast.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__)"],
    capture_output=True, text=True).stdout)

In [ ]:
# 4. Optionally mount Google Drive for the results
OUT = os.path.join(REPO, "exploratory", "bakeoff", "results")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = os.path.join(DRIVE_DIR, "results")
os.makedirs(OUT, exist_ok=True)
print("results folder:", OUT)

def run(args):
    """Run a command, stream its output, and stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="")
    if p.wait() != 0:
        raise RuntimeError("command failed: " + " ".join(args))

In [ ]:
# 5a. Monthly bake-off on the GPU (hash check and seasonal-naive gate run first)
run([sys.executable, "exploratory/bakeoff/run_bakeoff.py", "--resolution", "monthly", "--device", "cuda", "--out", OUT]
    + (["--models", *MODELS] if MODELS else []))

In [ ]:
# 5b. Daily bake-off on the GPU
run([sys.executable, "exploratory/bakeoff/run_bakeoff.py", "--resolution", "daily", "--device", "cuda", "--out", OUT]
    + (["--models", *MODELS] if MODELS else []))

In [ ]:
# 6. Build REPORT.md, zip results/ + REPORT.md, offer it as a download (and keep a copy in Drive)
import shutil
from IPython.display import Markdown, display
REPORT = os.path.join(os.path.dirname(OUT), "REPORT.md")
run([sys.executable, "exploratory/bakeoff/make_report.py", "--results", OUT, "--out", REPORT])
stage = "/content/bakeoff_export"
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
shutil.copytree(OUT, os.path.join(stage, "results"))
shutil.copy(REPORT, os.path.join(stage, "REPORT.md"))
zip_path = shutil.make_archive("/content/gtavi_bakeoff_results", "zip", stage)
if USE_DRIVE:
    shutil.copy(zip_path, DRIVE_DIR)
    print("copy saved to", os.path.join(DRIVE_DIR, os.path.basename(zip_path)))
from google.colab import files
files.download(zip_path)
display(Markdown(open(REPORT, encoding="utf-8").read()))